In [ ]:
# Floorplan AI Analyzer - YOLO Model Training and Testing
# This notebook trains a YOLOv8 model on floorplan data exported from Label Studio

In [ ]:
# Import necessary libraries
import os
import shutil
from pathlib import Path
import yaml
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [2]:
# Data setup and preparation
data_dir = Path('data')
images_dir = data_dir / 'images'
labels_dir = data_dir / 'labels'

# Read class names
with open(data_dir / 'classes.txt', 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

print(f"Classes: {class_names}")
print(f"Number of images: {len(list(images_dir.glob('*.png')))}")
print(f"Number of labels: {len(list(labels_dir.glob('*.txt')))}")

Classes: ['door', 'door_t2', 'stairs', 'window']
Number of images: 10
Number of labels: 10


In [3]:
# Split data into train/val/test sets
image_files = list(images_dir.glob('*.png'))
image_names = [f.stem for f in image_files]

# Split: 70% train, 20% val, 10% test
train_names, temp_names = train_test_split(image_names, test_size=0.3, random_state=42)
val_names, test_names = train_test_split(temp_names, test_size=0.33, random_state=42)

print(f"Train: {len(train_names)}, Val: {len(val_names)}, Test: {len(test_names)}")

# Create directories
for split in ['train', 'val', 'test']:
    (data_dir / split / 'images').mkdir(parents=True, exist_ok=True)
    (data_dir / split / 'labels').mkdir(parents=True, exist_ok=True)

# Move files
def move_files(names, split):
    for name in names:
        # Move image
        src_img = images_dir / f"{name}.png"
        dst_img = data_dir / split / 'images' / f"{name}.png"
        shutil.copy2(src_img, dst_img)
        
        # Move label
        src_label = labels_dir / f"{name}.txt"
        dst_label = data_dir / split / 'labels' / f"{name}.txt"
        if src_label.exists():
            shutil.copy2(src_label, dst_label)

move_files(train_names, 'train')
move_files(val_names, 'val')
move_files(test_names, 'test')

Train: 7, Val: 2, Test: 1


In [4]:
# Create data.yaml file for YOLO training
# Use absolute paths so YOLO does not prepend the workspace path twice
train_path = str((data_dir / 'train' / 'images').resolve())
val_path = str((data_dir / 'val' / 'images').resolve())
test_path = str((data_dir / 'test' / 'images').resolve())

data_yaml = {
    'train': train_path,
    'val': val_path,
    'test': test_path,
    'nc': len(class_names),
    'names': class_names
}


with open(data_dir / 'data.yaml', 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml created:")

print(yaml.dump(data_yaml, default_flow_style=False))

data.yaml created:
names:
- door
- door_t2
- stairs
- window
nc: 4
test: D:\For_Job\floorplan-ai-analyzer\data\test\images
train: D:\For_Job\floorplan-ai-analyzer\data\train\images
val: D:\For_Job\floorplan-ai-analyzer\data\val\images



In [14]:
# helper function to train a YOLO model, used by training cell and validation step

def train_yolo_model(epochs=10, imgsz=640, batch=8, name='floorplan_yolo', project='runs/train'):
    """Train a YOLO model using data.yaml in the data directory with error checking."""
    model = YOLO('yolov8n.pt')
    yaml_path = 'data/data.yaml'
    print(f"Starting training with data file {yaml_path}")
    print(f"Data file exists: {Path(yaml_path).exists()}")
    
    # Verify data.yaml content
    if Path(yaml_path).exists():
        with open(yaml_path, 'r') as f:
            print(f"data.yaml content preview:\n{f.read()[:200]}...")
    
    try:
        results = model.train(
            data=yaml_path,
            epochs=epochs,
            imgsz=imgsz,
            batch=batch,
            name=name,
            project=project,
            device='cpu'  # Use CPU for training
        )
        print(f"Training completed. Results: {results}")
        
        # Check where weights were saved
        weights_dir = Path(project) / name / 'weights'
        print(f"Looking for weights in: {weights_dir}")
        if weights_dir.exists():
            print(f"Weights directory contents: {list(weights_dir.glob('*'))}")
        else:
            print(f"Weights directory does not exist: {weights_dir}")
        
        return results
    except Exception as e:
        print(f"Training failed with error: {e}")
        raise

In [15]:
# Train YOLO model
# This cell calls the helper function defined earlier.  You can re-run it anytime to retrain.
# Using smaller batch size (4) for CPU training to avoid memory issues
results = train_yolo_model(epochs=10, imgsz=640, batch=4, name='floorplan_yolo', project='runs/train')
print("Training completed!")


Starting training with data file data/data.yaml
Data file exists: True
data.yaml content preview:
names:
- door
- door_t2
- stairs
- window
nc: 4
test: D:\For_Job\floorplan-ai-analyzer\data\test\images
train: D:\For_Job\floorplan-ai-analyzer\data\train\images
val: D:\For_Job\floorplan-ai-analyzer\...
Ultralytics 8.4.21  Python-3.14.3 torch-2.10.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, kera

2026/03/12 12:37:01 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


MLflow: logging run_id(b609efadc3ea4b5bb00bae251c65523a) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
WARNING MLflow: Failed to initialize: Changing param values is not allowed. Param with key='name' was already logged with value='floorplan_yolo7' for run ID='b609efadc3ea4b5bb00bae251c65523a'. Attempted logging new value 'floorplan_yolo8'.
WARNING MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 0 dataloader workers
Logging results to D:\For_Job\floorplan-ai-analyzer\runs\detect\runs\train\floorplan_yolo8
Starting training for 10 epochs...
Closing dataloader mosaic

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10         0G      1.852      4.425      1.411         23        640: 100% ━━━━━━━━━━━━ 2/2 3.8s/it 7.6s8.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━

In [17]:
# Validate the trained model
model_path = Path('D:/For_Job/floorplan-ai-analyzer/runs/detect/runs/train/floorplan_yolo8')
print(f"Looking for model at: {model_path.resolve()}")

if not model_path.exists():
    print(f"Model weights not found at {model_path}. Training now...")
    print("=" * 60)
    train_yolo_model(epochs=10, imgsz=640, batch=4, name='floorplan_yolo', project='runs/train')
    print("=" * 60)
    
    # Check again after training
    if not model_path.exists():
        # Try to find the weights in any subdirectory
        runs_dir = Path('runs/train')
        if runs_dir.exists():
            found_weights = list(runs_dir.glob('**/best.pt'))
            if found_weights:
                model_path = found_weights[0]
                print(f"Found weights at alternative location: {model_path}")
            else:
                print(f"No best.pt found in {runs_dir}")
                print(f"Directory contents: {list(runs_dir.rglob('*'))[:20]}")
                raise FileNotFoundError(f"Training completed but weights still missing at {model_path}.")
        else:
            raise FileNotFoundError(f"Training directory {runs_dir} does not exist.")

model = YOLO(str(model_path))

# Validate on validation set
metrics = model.val()
print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")

Looking for model at: D:\For_Job\floorplan-ai-analyzer\runs\detect\runs\train\floorplan_yolo8
Ultralytics 8.4.21  Python-3.14.3 torch-2.10.0+cpu CPU (Intel Core i5-10210U 1.60GHz)


TypeError: model='D:\For_Job\floorplan-ai-analyzer\runs\detect\runs\train\floorplan_yolo8' is not a supported model format. Ultralytics supports: ('PyTorch', 'TorchScript', 'ONNX', 'OpenVINO', 'TensorRT', 'CoreML', 'TensorFlow SavedModel', 'TensorFlow GraphDef', 'TensorFlow Lite', 'TensorFlow Edge TPU', 'TensorFlow.js', 'PaddlePaddle', 'MNN', 'NCNN', 'IMX', 'RKNN', 'ExecuTorch', 'Axelera')
See https://docs.ultralytics.com/modes/predict for help.

In [ ]:
# Debug: Check what's in the runs directory
import os
from pathlib import Path

runs_path = Path('runs/train')
print(f"Checking runs directory: {runs_path.resolve()}")
print(f"Directory exists: {runs_path.exists()}")

if runs_path.exists():
    print(f"\nContents of runs/train:")
    for item in runs_path.iterdir():
        print(f"  {item.name}/")
        if item.is_dir() and (item / 'weights').exists():
            weights_path = item / 'weights'
            print(f"    weights/")
            for weight_file in weights_path.glob('*'):
                print(f"      {weight_file.name}")
else:
    print("runs/train directory does not exist. Training may not have run.")


In [ ]:
# Test the model on test set
# Ensure the weights are available (validation cell already checked this)
test_images = list((data_dir / 'test' / 'images').glob('*.png'))

if not test_images:
    print("No test images found. Make sure the data split was performed correctly.")
else:
    results = model.predict(
        source=str(data_dir / 'test' / 'images'),
        save=True,
        save_txt=True,
        project='runs/test',
        name='predictions'
    )
    print("Testing completed! Results saved to runs/test/predictions/")

In [ ]:
# Visualize some results
import cv2

# Load a test image and its prediction
test_image_path = test_images[0] if test_images else None
if test_image_path:
    # Load original image
    img = cv2.imread(str(test_image_path))
    
    # Run prediction
    result = model.predict(str(test_image_path), save=False)[0]
    
    # Plot results
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB))
    plt.title(f"Prediction on {test_image_path.name}")
    plt.axis('off')
    plt.show()
    
    # Print detected objects
    for box in result.boxes:
        class_id = int(box.cls)
        confidence = box.conf.item()
        class_name = class_names[class_id]
        print(f"Detected: {class_name} with confidence {confidence:.2f}")